<a href="https://colab.research.google.com/github/JulianSantos-LATAMAI/ECON-5200/blob/main/lab_9/lab_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Aesthetic settings
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print("✅ Libraries loaded successfully.")

✅ Libraries loaded successfully.


In [4]:
# Load dataset
# If running in Colab, upload lalonde.csv first or mount your Drive.
# The dataset combines NSW treated units with PSID observational controls.
df = pd.read_csv('lalonde.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nTreatment group breakdown:")
print(df['treat'].value_counts().rename({0: 'Control (PSID)', 1: 'Treated (NSW)'}))
print(f"\nColumn overview:")
print(df.dtypes)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'lalonde.csv'

In [ ]:
# Load dataset
# If running in Colab, upload lalonde.csv first or mount your Drive.
# The dataset combines NSW treated units with PSID observational controls.
df = pd.read_csv('lalonde.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nTreatment group breakdown:")
print(df['treat'].value_counts().rename({0: 'Control (PSID)', 1: 'Treated (NSW)'}))
print(f"\nColumn overview:")
print(df.dtypes)
df.head()

In [2]:
treated_raw = df[df['treat'] == 1]
control_raw = df[df['treat'] == 0]

# Naive (unadjusted) difference in mean 1978 earnings
naive_diff = treated_raw['re78'].mean() - control_raw['re78'].mean()

print("=" * 45)
print("  NAIVE (UNADJUSTED) COMPARISON")
print("=" * 45)
print(f"  Treated mean re78 : ${treated_raw['re78'].mean():>10,.2f}")
print(f"  Control mean re78 : ${control_raw['re78'].mean():>10,.2f}")
print(f"  Naive Difference  : ${naive_diff:>10,.2f}")
print("=" * 45)
print()
print("⚠️  Insight: A negative estimate implies training HURTS")
print("   earnings — the hallmark of Selection Bias.")
print("   Controls are richer to begin with, making the")
print("   treated group look worse by comparison.")

NameError: name 'df' is not defined

In [ ]:
# Visualise the pre-matching imbalance on key confounders
confounders = ['age', 'educ', 're74', 're75']
confounder_labels = ['Age', 'Education (yrs)', 'Earnings 1974 ($)', 'Earnings 1975 ($)']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Pre-Matching Covariate Imbalance: Treated vs Control',
             fontsize=14, fontweight='bold', y=1.02)

for ax, col, label in zip(axes, confounders, confounder_labels):
    ax.hist(control_raw[col], bins=30, alpha=0.5, label='Control (PSID)', color='steelblue')
    ax.hist(treated_raw[col], bins=30, alpha=0.5, label='Treated (NSW)', color='tomato')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('pre_matching_imbalance.png', bbox_inches='tight')
plt.show()
print("\n📊 Notice how the distributions differ dramatically — this IS the selection bias.")

In [ ]:
# Define covariates (confounders affecting both treatment selection and outcome)
covariate_cols = ['age', 'educ', 'black', 'hisp', 'married', 'nodegree', 're74', 're75']

X = df[covariate_cols]
y = df['treat']

# Fit Propensity Score model
logit = LogisticRegression(solver='liblinear', max_iter=1000)
logit.fit(X, y)

# Generate propensity scores
df['pscore'] = logit.predict_proba(X)[:, 1]

print("Propensity Score Model Summary")
print("=" * 40)
coef_df = pd.DataFrame({
    'Covariate': covariate_cols,
    'Coefficient': logit.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df.to_string(index=False))
print()
print(f"Propensity score range: [{df['pscore'].min():.4f}, {df['pscore'].max():.4f}]")
print(f"Mean p-score (treated) : {df.loc[df.treat==1, 'pscore'].mean():.4f}")
print(f"Mean p-score (control) : {df.loc[df.treat==0, 'pscore'].mean():.4f}")

In [ ]:
# Visualise propensity score distributions before matching
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(df.loc[df.treat==0, 'pscore'], bins=50, alpha=0.55,
        label='Control (PSID)', color='steelblue', density=True)
ax.hist(df.loc[df.treat==1, 'pscore'], bins=50, alpha=0.55,
        label='Treated (NSW)', color='tomato', density=True)

ax.set_xlabel('Propensity Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Propensity Score Distribution Before Matching', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('pscore_distribution.png', bbox_inches='tight')
plt.show()
print("\n📊 Severe overlap issue visible — most controls have near-zero p-scores.")

In [ ]:
# Separate groups
treated = df[df['treat'] == 1].copy().reset_index(drop=True)
control = df[df['treat'] == 0].copy().reset_index(drop=True)

# Fit NearestNeighbors on control propensity scores
nbrs = NearestNeighbors(n_neighbors=1, algorithm='ball_tree', metric='euclidean')
nbrs.fit(control[['pscore']])

# Find nearest control match for each treated unit
distances, indices = nbrs.kneighbors(treated[['pscore']])

# Retrieve matched control rows (with replacement — duplicates allowed)
matched_control = control.iloc[indices.flatten()].copy().reset_index(drop=True)

# Construct matched dataset
matched_df = pd.concat([treated, matched_control], ignore_index=True)

print(f"Treated units          : {len(treated)}")
print(f"Matched control units  : {len(matched_control)} (with replacement)")
print(f"Unique controls used   : {len(set(indices.flatten()))}")
print(f"Total matched sample   : {len(matched_df)}")
print(f"\nMean match distance    : {distances.mean():.6f}")
print(f"Max match distance     : {distances.max():.6f}")

In [ ]:
def smd(col, treated_df, control_df):
    """Compute Standardized Mean Difference for a single covariate."""
    t_mean, c_mean = treated_df[col].mean(), control_df[col].mean()
    t_var,  c_var  = treated_df[col].var(),  control_df[col].var()
    pooled_sd = np.sqrt((t_var + c_var) / 2)
    if pooled_sd == 0:
        return 0.0
    return (t_mean - c_mean) / pooled_sd

# Compare SMD before and after matching
matched_treated_df = matched_df[matched_df['treat'] == 1]
matched_control_df = matched_df[matched_df['treat'] == 0]

balance = pd.DataFrame({
    'Covariate': covariate_cols,
    'SMD (Before)': [smd(c, treated, control) for c in covariate_cols],
    'SMD (After)':  [smd(c, matched_treated_df, matched_control_df) for c in covariate_cols]
})

balance['|SMD| After'] = balance['SMD (After)'].abs()
balance['Balanced?']   = balance['|SMD| After'] < 0.1

print("COVARIATE BALANCE ASSESSMENT")
print("=" * 62)
print(balance[['Covariate', 'SMD (Before)', 'SMD (After)', 'Balanced?']].to_string(index=False))
print("=" * 62)
n_balanced = balance['Balanced?'].sum()
print(f"\n✅ {n_balanced}/{len(covariate_cols)} covariates meet the |SMD| < 0.1 threshold.")

In [ ]:
# Love Plot — visualise SMD improvement
fig, ax = plt.subplots(figsize=(9, 5))

y_pos = np.arange(len(covariate_cols))
ax.barh(y_pos - 0.2, balance['SMD (Before)'].abs(), height=0.35,
        label='Before Matching', color='tomato', alpha=0.75)
ax.barh(y_pos + 0.2, balance['SMD (After)'].abs(),  height=0.35,
        label='After Matching',  color='seagreen', alpha=0.75)

ax.axvline(0.1, color='black', linestyle='--', linewidth=1.2, label='Balance Threshold (0.1)')
ax.set_yticks(y_pos)
ax.set_yticklabels(covariate_cols, fontsize=11)
ax.set_xlabel('Absolute Standardized Mean Difference', fontsize=12)
ax.set_title('Love Plot: Covariate Balance Before and After Matching',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('love_plot.png', bbox_inches='tight')
plt.show()

In [ ]:
# Propensity score overlap after matching
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (t_df, c_df), title in zip(
    axes,
    [(treated, control), (matched_treated_df, matched_control_df)],
    ['Before Matching', 'After Matching']
):
    ax.hist(c_df['pscore'], bins=40, alpha=0.5, label='Control', color='steelblue', density=True)
    ax.hist(t_df['pscore'], bins=40, alpha=0.5, label='Treated', color='tomato',    density=True)
    ax.set_title(f'P-score Distribution — {title}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Propensity Score')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Propensity Score Overlap: Before vs After Matching', fontsize=13, y=1.03)
plt.tight_layout()
plt.savefig('pscore_overlap.png', bbox_inches='tight')
plt.show()

In [3]:
# ── Raw (unadjusted) comparison ──────────────────────────────────────────────
diff_raw = treated['re78'].mean() - control['re78'].mean()
t_stat_raw, p_val_raw = stats.ttest_ind(treated['re78'], control['re78'])

# ── Matched comparison ───────────────────────────────────────────────────────
matched_t_re78 = matched_df[matched_df['treat'] == 1]['re78']
matched_c_re78 = matched_df[matched_df['treat'] == 0]['re78']

matched_diff = matched_t_re78.mean() - matched_c_re78.mean()
t_stat_match, p_val_match = stats.ttest_ind(matched_t_re78, matched_c_re78)

# ── RCT benchmark ───────────────────────────────────────────────────────────
rct_benchmark = 1794  # LaLonde (1986) experimental estimate

print("╔══════════════════════════════════════════════════════╗")
print("║           TREATMENT EFFECT ESTIMATES                 ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Raw (Unadjusted) Difference : ${diff_raw:>9,.2f}              ║")
print(f"║  T-statistic                 : {t_stat_raw:>9.4f}              ║")
print(f"║  P-value                     : {p_val_raw:>9.4f}              ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Matched (PSM) Difference    : ${matched_diff:>9,.2f}              ║")
print(f"║  T-statistic                 : {t_stat_match:>9.4f}              ║")
print(f"║  P-value                     : {p_val_match:>9.4f}              ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  RCT Benchmark (LaLonde '86) : ${rct_benchmark:>9,}              ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Bias Removed by PSM         : ${abs(matched_diff - diff_raw):>9,.2f}              ║")
print(f"║  Remaining Bias vs RCT       : ${abs(matched_diff - rct_benchmark):>9,.2f}              ║")
print("╚══════════════════════════════════════════════════════╝")

NameError: name 'treated' is not defined

In [ ]:
# Summary bar chart: comparing all three estimates
labels   = ['Naive\n(Unadjusted)', 'PSM\n(Matched)', 'RCT Benchmark\n(LaLonde 1986)']
values   = [diff_raw, matched_diff, rct_benchmark]
colors   = ['tomato', 'steelblue', 'seagreen']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, values, color=colors, alpha=0.82, edgecolor='white', linewidth=1.2)

for bar, val in zip(bars, values):
    ypos = val + 30 if val >= 0 else val - 130
    ax.text(bar.get_x() + bar.get_width() / 2, ypos,
            f'${val:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Estimated Treatment Effect on re78 ($)', fontsize=12)
ax.set_title('Recovery of the Causal Effect via Propensity Score Matching',
             fontsize=13, fontweight='bold')
ax.set_ylim(min(values) * 1.3, max(values) * 1.25)

plt.tight_layout()
plt.savefig('effect_comparison.png', bbox_inches='tight')
plt.show()